In [4]:
import pandas as pd

pd.read_hdf('./data_base_GES1point5/data_base_GES1point5.hdf5')

,category,subcategory,subsubcategory,unit,name,year,co2,ch4,n2o,other,total,uncertainty,ef.unit
0,Électricité,FR,NaN,kWh,Electricité France continentale,2022,0.0,0.0,0.0,0,0.0520,0.10,kg eCO2/kWh
1,Électricité,FR.94,NaN,kWh,Electricité Corse,2014,0.0,0.0,0.0,0,0.5937,0.15,kg eCO2/kWh
2,Électricité,PM,NaN,kWh,Electricité St Pierre et Miquelon,2017,0.0,0.0,0.0,0,0.9438,0.15,kg eCO2/kWh
3,Électricité,WF,NaN,kWh,Electricité Wallis-et-Futuna (identique à St P...,2017,0.0,0.0,0.0,0,0.9438,0.15,kg eCO2/kWh
4,Électricité,TF,NaN,kWh,Electricité Terres australes françaises (ident...,2017,0.0,0.0,0.0,0,0.9438,0.15,kg eCO2/kWh
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1635,Véhicules,bus,bus.intercity,km,Autocar - trajets intercités,2021,0.0,0.0,0.0,0,0.0306,0.60,kg eCO2/km
1636,Véhicules,bus,Bus > 250 000 habitants,km,Autobus moyen - Agglomération de plus de 250 0...,2021,0.0,0.0,0.0,0,0.1290,0.60,kg eCO2/km
1637,Véhicules,bus,Bus 100 000 - 250 000 habitants,km,Autobus moyen - Agglomération de 100 000 à 250...,2021,0.0,0.0,0.0,0,0.1370,0.60,kg eCO2/km
1638,Véhicules,bus,Bus < 100 000 habitants,km,Autobus moyen - Agglomération moins de 100 000...,2021,0.0,0.0,0.0,0,0.1460,0.60,kg eCO2/km


In [5]:
"""
Convertit le fichier Excel PER1p5 (NACRES–EF) en un HDF5 exploitable par LABeCO2,
en conservant *toutes* les colonnes utiles (même si non utilisées tout de suite).

- Lit l’onglet "NACRES-EF" (et garde l’onglet README à part si présent)
- Nettoie les noms de colonnes, force les types "codes/labels" en str (dtype object, compat HDF5)
- Produit 2 tables HDF5 :
    1) /raw_nacres_ef       : table quasi brute (nettoyée)
    2) /purchases_factors   : table "compat" proche de ton TSV historique (Achats)
- Exporte aussi 2 TSV (optionnel mais pratique pour diff/inspection)

Usage (script):
    python convert_per1p5_excel_to_hdf5.py

Usage (notebook):
    main()
"""

from __future__ import annotations

from pathlib import Path
import pandas as pd


# --------------------------
# PARAMÈTRES (à adapter)
# --------------------------
EXCEL_PATH = Path(
    "/Users/souchaud/Documents/Travail/Transition_ecologique/LABeCO2/"
    "data_base_GES1point5/2024_dataverse_files/PER1p5_nacres_fe_database_v1-0-2023.xlsx"
)

# ⚠️ mets ici le dossier où tu veux générer les fichiers (ex: data_base_GES1point5/data_initiales/)
OUT_DIR = Path(".")

OUT_H5 = OUT_DIR / "GES1point5_purchases_factors_PER1p5_v1-0-2023.h5"
OUT_TSV_RAW = OUT_DIR / "PER1p5_nacres_ef_raw_v1-0-2023.tsv"
OUT_TSV_COMPAT = OUT_DIR / "GES1point5_purchases_factors_PER1p5_compat_v1-0-2023.tsv"

SHEET_DATA = "NACRES-EF"
SHEET_README = "README"  # facultatif

DEFAULT_YEAR = 2019
DEFAULT_UNIT = "euro"
DEFAULT_EF_UNIT = "kg eCO2/euro"


# --------------------------
# OUTILS
# --------------------------
def _clean_colnames(cols) -> list[str]:
    """Supprime les espaces parasites sans changer les points '.' (utile si ton code s'appuie dessus)."""
    return [str(c).strip() for c in cols]


def _force_object_str(df: pd.DataFrame, cols: list[str]) -> None:
    """
    Force certaines colonnes à être des chaînes Python (dtype object) + strip,
    tout en gardant les NaN.
    => compatible HDFStore(format="table")
    """
    for c in cols:
        if c in df.columns:
            s = df[c]
            # Convertit en object, puis strip seulement sur valeurs non nulles
            s = s.astype(object)
            s = s.where(pd.isna(s), s.astype(str).str.strip())
            df[c] = s


def _to_float(df: pd.DataFrame, cols: list[str]) -> None:
    """Convertit en float quand possible (NaN sinon)."""
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")


def _category_fr_from_per1p5(cat: object) -> str:
    """Mapping PER1p5 'category' -> libellé FR proche de ton TSV historique."""
    mapping = {
        "lab.life": "Vie du laboratoire (Alimentation, aménagement, loisirs, bâtiment)",
        "services": "Services",
        "transport": "Transport / Hébergement",
        "consumables": "Consommables (Matières premières, produits chimiques/biologiques et organismes vivants)",
        "lab.equipment": "Matériel et instruments de laboratoire",
        "maintenance": "Réparations et maintenance",
        "info": "Informatique-audiovisuel",
    }
    if cat is None or (isinstance(cat, float) and pd.isna(cat)) or (isinstance(cat, str) and cat.strip() == ""):
        return "Autre"
    cat_str = str(cat).strip()
    return mapping.get(cat_str, cat_str)


def _ensure_hdf_compat(df: pd.DataFrame) -> pd.DataFrame:
    """
    Assure une compatibilité maximale avec HDFStore(format="table"):
    - Toutes les colonnes dtype 'string[python]' (pandas StringDtype) => object
    - Rien à faire si déjà object/float/int/bool/datetime
    """
    out = df.copy()
    for c in out.columns:
        if pd.api.types.is_string_dtype(out[c].dtype):
            out[c] = out[c].astype(object)
    return out


def _min_itemsize_for_text_columns(df: pd.DataFrame) -> dict[str, int]:
    """
    Calcule des tailles minimales pour les colonnes texte, pour éviter
    des erreurs PyTables sur chaînes longues.
    """
    min_itemsize: dict[str, int] = {}
    for c in df.columns:
        if df[c].dtype == object:
            # ne regarde que des str
            try:
                lengths = df[c].dropna().astype(str).map(len)
                if len(lengths) > 0:
                    mx = int(lengths.max())
                    # marge + plafonds raisonnables
                    if mx > 50:
                        min_itemsize[c] = min(mx + 20, 500)
            except Exception:
                # si colonne hétérogène, on ignore
                pass
    return min_itemsize


# --------------------------
# PIPELINE
# --------------------------
def main() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # --- Lecture Excel
    xls = pd.ExcelFile(EXCEL_PATH)
    if SHEET_DATA not in xls.sheet_names:
        raise ValueError(f"Onglet '{SHEET_DATA}' introuvable. Onglets: {xls.sheet_names}")

    df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_DATA, dtype=object)
    df.columns = _clean_colnames(df.columns)

    # --- Nettoyage minimal (types)
    str_cols = [
        "nacres.code",
        "nacres.description.fr",
        "nacres.description.en",
        "method",
        "module",
        "category",
        "meso.code",
        "meso.description",
        "micro.code",
        "micro.description",
        "ceda.code",
        "ceda.description",
        "ademe.code",
        "ademe.description",
        "useeio.code",
        "useeio.description",
        "uncertainty.groupby",
        "scope.source.v4",
        "scope.source.v5",
    ]
    _force_object_str(df, str_cols)

    num_cols = [
        "per1p5.ef.kg.co2e.per.euro",
        "per1p5.uncertainty.attr.kg.co2e.per.euro",
        "per1p5.uncertainty.80pct.kg.co2e.per.euro",
        "per1p5macro.ef.kg.co2e.per.euro",
        "per1p5macro.uncertainty.attr.kg.co2e.per.euro",
        "per1p5macro.uncertainty.80pct.kg.co2e.per.euro",
        "meso.ef.kg.co2e.per.euro",
        "meso.uncertainty.kg.co2e.per.euro",
        "micro.ef.kg.co2e.per.euro",
        "micro.uncertainty.kg.co2e.per.euro",
        "ademe.ef.kg.co2e.per.euro",
        "ademe.uncertainty.attr.kg.co2e.per.euro",
        "ademe.uncertainty.80pct.kg.co2e.per.euro",
        "useeio.ef.kg.co2e.per.euro",
        "useeio.uncertainty.attr.kg.co2e.per.euro",
        "useeio.uncertainty.80pct.kg.co2e.per.euro",
        "ceda.ef.kg.co2e.per.euro",
        "ceda.uncertainty.attr.kg.co2e.per.euro",
        "ceda.uncertainty.80pct.kg.co2e.per.euro",
    ]
    _to_float(df, num_cols)

    # --- Filtrage : garder uniquement les lignes avec nacres.code
    if "nacres.code" not in df.columns:
        raise ValueError("Colonne 'nacres.code' manquante dans l'onglet NACRES-EF.")
    df = df[df["nacres.code"].notna() & (df["nacres.code"].astype(str).str.len() > 0)].copy()

    # --- Export TSV brut (inspection)
    df.to_csv(OUT_TSV_RAW, sep="\t", index=False)

    # --- Construction table "compat" (proche de ton TSV historique)
    compat = pd.DataFrame()
    compat["category"] = "Achats"
    compat["subcategory"] = df["category"].apply(_category_fr_from_per1p5)
    compat["subsubcategory"] = df["nacres.code"]
    compat["unit"] = DEFAULT_UNIT
    compat["name"] = df["nacres.description.fr"].where(
        df["nacres.description.fr"].notna() & (df["nacres.description.fr"].astype(str).str.len() > 0),
        df["nacres.description.en"],
    )
    compat["year"] = DEFAULT_YEAR

    # Structure gaz (ton format historique)
    compat["co2"] = 0.0
    compat["ch4"] = 0.0
    compat["n2o"] = 0.0
    compat["other"] = 0.0

    # total : facteur final arbitrée PER1p5
    if "per1p5.ef.kg.co2e.per.euro" not in df.columns:
        raise ValueError("Colonne 'per1p5.ef.kg.co2e.per.euro' manquante.")
    compat["total"] = df["per1p5.ef.kg.co2e.per.euro"]

    # uncertainty : incertitude absolue (80%)
    u80 = "per1p5.uncertainty.80pct.kg.co2e.per.euro"
    compat["uncertainty"] = df[u80] if u80 in df.columns else pd.NA

    compat["ef.unit"] = DEFAULT_EF_UNIT

    # Colonnes extra (audit/affichage)
    for c in [
        "method",
        "module",
        "nacres.description.en",
        "per1p5macro.ef.kg.co2e.per.euro",
        "meso.ef.kg.co2e.per.euro",
        "micro.ef.kg.co2e.per.euro",
        "per1p5.uncertainty.attr.kg.co2e.per.euro",
        "micro.code",
        "micro.description",
        "meso.code",
        "meso.description",
        "uncertainty.groupby",
    ]:
        if c in df.columns:
            compat[c] = df[c]

    # Forcer ces nouvelles colonnes en object str là où pertinent
    compat_str_cols = [
        "category", "subcategory", "subsubcategory", "unit", "name", "ef.unit",
        "method", "module", "nacres.description.en", "micro.code", "micro.description",
        "meso.code", "meso.description", "uncertainty.groupby"
    ]
    _force_object_str(compat, [c for c in compat_str_cols if c in compat.columns])

    # Export TSV compat
    compat.to_csv(OUT_TSV_COMPAT, sep="\t", index=False)

    # --- Écriture HDF5 (compat PyTables)
    df_h5 = _ensure_hdf_compat(df)
    compat_h5 = _ensure_hdf_compat(compat)

    # Astuce PyTables : prévoir des tailles de chaînes pour colonnes texte longues
    min_itemsize_raw = _min_itemsize_for_text_columns(df_h5)
    min_itemsize_compat = _min_itemsize_for_text_columns(compat_h5)

    with pd.HDFStore(OUT_H5, mode="w") as store:
        store.put(
            "raw_nacres_ef",
            df_h5,
            format="table",
            data_columns=True,
            min_itemsize=min_itemsize_raw if min_itemsize_raw else None,
        )
        store.put(
            "purchases_factors",
            compat_h5,
            format="table",
            data_columns=True,
            min_itemsize=min_itemsize_compat if min_itemsize_compat else None,
        )

        # Optionnel : stocker l'onglet README si présent
        if SHEET_README in xls.sheet_names:
            readme = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_README, dtype=object)
            readme.columns = _clean_colnames(readme.columns)
            readme = _ensure_hdf_compat(readme)
            mi_readme = _min_itemsize_for_text_columns(readme)
            store.put(
                "readme",
                readme,
                format="table",
                data_columns=True,
                min_itemsize=mi_readme if mi_readme else None,
            )

    print("OK ✅")
    print(f"- HDF5 : {OUT_H5}")
    print(f"- TSV brut : {OUT_TSV_RAW}")
    print(f"- TSV compat : {OUT_TSV_COMPAT}")
    print("Tables HDF5 écrites : /raw_nacres_ef , /purchases_factors (et /readme si présent).")


# Pour exécuter depuis un notebook:
# main()

In [6]:

if __name__ == "__main__":
    main()

/Users/souchaud/Documents/Travail/Transition_ecologique/LABeCO2/LABeCO2_env_test/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/souchaud/Documents/Travail/Transition_ecologique/LABeCO2/LABeCO2_env_test/lib/python3.11/site-packages/tables/path.py:137: NaturalNameWarning: object name is not a valid Python identifier: 'nacres.code'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though
  check_attribute_name(name)
/Users/souchaud/Documents/Travail/Transition_ecologique/LABeCO2/LABeCO2_env_test/lib/python3.11/site-packages/tables/path.py:137: NaturalNameWarning: object name is not a valid Python identifier: 'nacres.description.fr'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`

OK ✅
- HDF5 : GES1point5_purchases_factors_PER1p5_v1-0-2023.h5
- TSV brut : PER1p5_nacres_ef_raw_v1-0-2023.tsv
- TSV compat : GES1point5_purchases_factors_PER1p5_compat_v1-0-2023.tsv
Tables HDF5 écrites : /raw_nacres_ef , /purchases_factors (et /readme si présent).


/Users/souchaud/Documents/Travail/Transition_ecologique/LABeCO2/LABeCO2_env_test/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/souchaud/Documents/Travail/Transition_ecologique/LABeCO2/LABeCO2_env_test/lib/python3.11/site-packages/tables/path.py:137: NaturalNameWarning: object name is not a valid Python identifier: 'Purchases emissions in research 1point5 NACRES-EF dataset'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though
  check_attribute_name(name)
/Users/souchaud/Documents/Travail/Transition_ecologique/LABeCO2/LABeCO2_env_test/lib/python3.11/site-packages/tables/path.py:137: NaturalNameWarning: object name is not a valid Python identifier: 'Unnamed: 1'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to acc

In [10]:
import pandas as pd

store = pd.HDFStore("./GES1point5_purchases_factors_PER1p5_v1-0-2023.h5", mode="r")
print(store.keys())
store.close()

['/purchases_factors', '/raw_nacres_ef', '/readme']


In [11]:
store

<class 'pandas.io.pytables.HDFStore'>
File path: ./GES1point5_purchases_factors_PER1p5_v1-0-2023.h5

In [14]:
df_raw = pd.read_hdf("./GES1point5_purchases_factors_PER1p5_v1-0-2023.h5", key="raw_nacres_ef")
df_raw.columns

Index(['nacres.code', 'nacres.description.fr', 'nacres.description.en',
       'ceda.code', 'ceda.description', 'ceda.ef.kg.co2e.per.euro',
       'ceda.uncertainty.attr.kg.co2e.per.euro',
       'ceda.uncertainty.80pct.kg.co2e.per.euro', 'ademe.code',
       'ademe.description', 'ademe.ef.kg.co2e.per.euro',
       'ademe.uncertainty.attr.kg.co2e.per.euro',
       'ademe.uncertainty.80pct.kg.co2e.per.euro', 'useeio.code',
       'useeio.description', 'useeio.ef.kg.co2e.per.euro',
       'useeio.uncertainty.attr.kg.co2e.per.euro',
       'useeio.uncertainty.80pct.kg.co2e.per.euro',
       'per1p5macro.ef.kg.co2e.per.euro',
       'per1p5macro.uncertainty.attr.kg.co2e.per.euro',
       'per1p5macro.uncertainty.80pct.kg.co2e.per.euro', 'meso.code',
       'meso.description', 'meso.ef.kg.co2e.per.euro',
       'meso.uncertainty.kg.co2e.per.euro', 'micro.code', 'micro.description',
       'micro.ef.kg.co2e.per.euro', 'micro.uncertainty.kg.co2e.per.euro',
       'method', 'per1p5.ef.kg.co2e.

In [17]:
pd.read_hdf("./data_base_GES1point5.hdf5").columns

Index(['category', 'subcategory', 'subsubcategory', 'unit', 'name', 'year',
       'co2', 'ch4', 'n2o', 'other', 'total', 'uncertainty', 'ef.unit'],
      dtype='object')

In [6]:
import pandas as pd

# --------------------------
# PARAMÈTRES
# --------------------------
INPUT_TSV = "GES1point5_purchases_factors_PER1p5_compat_v1-0-2023.tsv"
OUTPUT_TSV = "GES1point5_purchases_micro_only.tsv"

# --------------------------
# LECTURE DU FICHIER
# --------------------------
df = pd.read_csv(INPUT_TSV, sep="\t")

# Vérification minimale
if "method" not in df.columns:
    raise ValueError("La colonne 'method' est absente du fichier TSV.")

# --------------------------
# FILTRAGE : uniquement micro
# --------------------------
df_micro = df[df["method"] == "micro"].copy()

# --------------------------
# (OPTIONNEL) Sélection de colonnes utiles
# Commente cette section si tu veux TOUT garder
# --------------------------
cols_keep = [
    "category",
    "subcategory",
    "subsubcategory",
    "unit",
    "name",
    "year",
    "total",
    "uncertainty",
    "ef.unit",
    "method",
]

# Garde uniquement les colonnes existantes
cols_keep = [c for c in cols_keep if c in df_micro.columns]
df_micro = df_micro[cols_keep]

# --------------------------
# EXPORT
# --------------------------
# df_micro.to_csv(OUTPUT_TSV, sep="\t", index=False)

# --------------------------
# INFO
# --------------------------
print("Extraction terminée ✅")
print(f"- Fichier source : {INPUT_TSV}")
print(f"- Fichier micro  : {OUTPUT_TSV}")
print(f"- Nombre de lignes micro : {len(df_micro)}")

Extraction terminée ✅
- Fichier source : GES1point5_purchases_factors_PER1p5_compat_v1-0-2023.tsv
- Fichier micro  : GES1point5_purchases_micro_only.tsv
- Nombre de lignes micro : 31


In [7]:
df_micro

,category,subcategory,subsubcategory,unit,name,year,total,uncertainty,ef.unit,method
417,NaN,"Consommables (Matières premières, produits chi...",GA01,euro,ACETYLENE DE QUALITE INDUSTRIELLE EN BOUTEILLE,2019,0.67,0.536,kg eCO2/euro,micro
418,NaN,"Consommables (Matières premières, produits chi...",GA02,euro,AIR SYNTHETIQUE DE QUALITE INDUSTRIELLE EN BOU...,2019,0.22,0.176,kg eCO2/euro,micro
419,NaN,"Consommables (Matières premières, produits chi...",GA03,euro,ARGON DE QUALITE INDUSTRIELLE EN BOUTEILLE,2019,0.39,0.312,kg eCO2/euro,micro
420,NaN,"Consommables (Matières premières, produits chi...",GA04,euro,AZOTE GAZEUX DE QUALITE INDUSTRIELLE EN BOUTEILLE,2019,0.17,0.136,kg eCO2/euro,micro
421,NaN,"Consommables (Matières premières, produits chi...",GA05,euro,DIOXYDE DE CARBONE DE QUALITE INDUSTRIELLE EN ...,2019,0.47,0.376,kg eCO2/euro,micro
422,NaN,"Consommables (Matières premières, produits chi...",GA06,euro,HELIUM GAZEUX DE QUALITE INDUSTRIELLE EN BOUTE...,2019,0.07,0.056,kg eCO2/euro,micro
423,NaN,"Consommables (Matières premières, produits chi...",GA07,euro,HYDROGENE DE QUALITE INDUSTRIELLE EN BOUTEILLE,2019,0.45,0.360,kg eCO2/euro,micro
424,NaN,"Consommables (Matières premières, produits chi...",GA08,euro,OXYGENE DE QUALITE INDUSTRIELLE EN BOUTEILLE,2019,0.32,0.256,kg eCO2/euro,micro
425,NaN,"Consommables (Matières premières, produits chi...",GA09,euro,AUTRES GAZ SIMPLES OU EN MELANGE DE QUALITE IN...,2019,0.35,0.280,kg eCO2/euro,micro
426,NaN,"Consommables (Matières premières, produits chi...",GA11,euro,ARGON DE TRES HAUTE PURETE (SUPERIEURE A 5.0) ...,2019,0.31,0.248,kg eCO2/euro,micro
